[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_70_Launch_Day_AgentBench.ipynb)

# Lesson 70 — Launch Day: Shipping `agent-bench` v0.1.0 🚀
### Phase 7 Capstone · Your **second** flagship OSS tool goes public

> **Where we are.** Over Phase 7 (L65–L69) you rebuilt `agent-bench` from a notebook demo
> (L61) into a real, installable, plugin-extensible, async, CLI-driven, PyPI-packaged,
> README-and-CONTRIBUTING-documented Python library. Every piece was validated *in its own
> lesson's fixtures*. Today is the day they run together in one process, get packaged as a
> real wheel, and — for the parts that require **your** GitHub/PyPI identity — get a precise,
> copy-pasteable runbook.

**This is a capstone, not a new-concept lesson.** The single genuinely new idea is
*launching the second repo in a portfolio* — coordinating `agent-bench`'s launch against
`paper-distiller` (shipped in L64) so the two cross-promote instead of cannibalizing each
other. Everything else is integration, verification, and the honest boundary of what an
assistant can and cannot do on your behalf.

---
### Phase 7 roadmap (L65 → L70)

| # | Lesson | Status |
|---|--------|--------|
| 65 | Phase 7 kickoff — `agent-bench` as a 2nd flagship, plugin-registry pattern | ✅ |
| 66 | Real plugins — `ShellEnv` + `importlib.metadata.entry_points()` | ✅ |
| 67 | Parallel execution — `asyncio.gather` + `Semaphore` | ✅ |
| 68 | CLI polish + PyPI packaging | ✅ |
| 69 | OSS growth round two — README, badges, contributor funnel, cross-promotion | ✅ |
| **70** | **Launch Day — ship `agent-bench` v0.1.0 (capstone)** | **← you are here** |

By the end of this notebook `agent-bench` will be a wheel that passes `twine check`, with a
launch runbook you can execute the moment you're ready.


## 1 · "Code that works" vs. "a shipped project" — *round two*

You saw this table in L64 for `paper-distiller`. Re-reading it now, for a **second** repo,
the interesting column is the last one: what changes when you already have one public project.

| Dimension | Code that works | A shipped project | *What's different the 2nd time* |
|-----------|-----------------|-------------------|-------------------------------|
| **Discovery** | Lives in a notebook | On PyPI + GitHub, `pip install agent-bench` | Your first repo's README can *link* here |
| **Installability** | Copy-paste cells | One command, pinned deps | Reuse L68's `pyproject.toml` shape verbatim |
| **Composability** | One monolith | Stable core + plugins | Plugins can span *both* repos |
| **Trust** | "works on my machine" | Green CI, LICENSE, tests | A second green repo compounds credibility |
| **First contact** | You explain it in DMs | README does the talking | Cross-promo callout in *both* READMEs |
| **Launch risk** | n/a | Botched launch = dead repo | **NEW: don't launch two repos the same week** |

The load-bearing risk today is the same one L64 flagged and L69 named explicitly: modules
validated *in isolation* silently diverge. `core.py`, `registry.py`, `environments.py`,
`runner_async.py`, and `cli.py` were each last touched in a different lesson. **This notebook
runs them together, in one kernel, for the first time.**


## 2 · Setup

No `ANTHROPIC_API_KEY` needed — the whole harness runs on a deterministic `MockAgent`, so
launch-day verification is free and reproducible. (A real Claude agent is a one-line swap,
covered back in L61/L67.)


In [ ]:
import subprocess, sys

def pip_install(pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

# Lightweight launch-day deps only — no chromadb/sentence-transformers here.
pip_install(["typer", "pydantic", "rich", "nest_asyncio", "build", "twine"])

import os, io, json, textwrap, pathlib, asyncio, importlib, shutil, sysconfig, site
import nest_asyncio
nest_asyncio.apply()   # notebooks already run an event loop; agent-bench's run_sync calls asyncio.run

ROOT = "/content"                        # real Colab path; validation shim may redirect this
PKG_DIR = f"{ROOT}/agent_bench_pkg"       # the installable package tree
os.makedirs(PKG_DIR, exist_ok=True)
print("Setup OK. Building agent-bench under:", PKG_DIR)


## 3 · Final architecture — everything wired into one tree

This is the whole package as it stands after L65–L69, consolidated. `core.py` never imports
from the leaf modules; the leaves import from `core` and *register themselves* into the
registries (the plugin pattern from L65).

```
agent_bench_pkg/
├── pyproject.toml              # L68 — hatchling, [project.scripts], entry-points table
├── README.md                  # L69 — badges, cross-promo to paper-distiller, plugin gallery
├── CONTRIBUTING.md            # L69 — PR-to-core vs publish-a-plugin funnel
├── LICENSE                    # MIT
├── .github/
│   ├── workflows/ci.yml       # L68 — matrix test + build + twine check
│   ├── workflows/release.yml  # L68 — tag-triggered OIDC Trusted Publishing
│   └── ISSUE_TEMPLATE/        # L69 — bug/feature/plugin_submission forms
└── agent_bench/
    ├── __init__.py
    ├── core.py                # L61/L65 — Task, Trajectory, TaskResult, pass_at_k
    ├── registry.py            # L65 core + L66 plugin discovery
    ├── environments.py        # L67 — CalcEnv, FileEnv, MockAgent, scorers
    ├── runner_async.py        # L67 — AsyncBenchmarkRunner (run_sync / run_async)
    └── cli.py                 # L68 — Typer app, --version, --format
```

Design decisions, one table (the same discipline as L64):

| Decision | Why |
|----------|-----|
| Registries store **classes**, not instances | L67 — fresh env per task, no cross-task state leak |
| Plugin discovery is **read-then-load**, failures per-plugin | L66 — one broken 3rd-party package can't crash the harness |
| CLI is a thin wrapper, zero business logic | L65/L68 — testable core, swappable frontend |
| `run_sync` wraps `run_async` via `asyncio.run` | L67 — one code path, sync convenience for scripts |


### 3.1 · `core.py` — unchanged since L61/L65

Task / Trajectory / TaskResult data models + the `pass_at_k` unbiased estimator. Recreated
verbatim; not re-litigated (already validated across four lessons).


In [ ]:
core_py = r"""
from __future__ import annotations
import math
from dataclasses import dataclass, field
from typing import Any, Callable
from pydantic import BaseModel

class Task(BaseModel):
    id: str
    category: str
    difficulty: str
    prompt: str
    env_name: str
    scorer_name: str
    answer: str | None = None

@dataclass
class TrajectoryStep:
    tool: str
    args: dict
    observation: Any

@dataclass
class Trajectory:
    steps: list = field(default_factory=list)
    final_state: Any = None
    def add(self, tool, args, observation):
        self.steps.append(TrajectoryStep(tool, args, observation))

@dataclass
class TaskResult:
    task_id: str
    passed: bool
    score: float
    error: str | None = None

def pass_at_k(n: int, c: int, k: int) -> float:
    # Unbiased estimator (Chen et al. 2021). n=attempts, c=correct, k=budget.
    if n - c < k:
        return 1.0
    return 1.0 - math.prod((n - c - i) / (n - i) for i in range(k))
"""
import os
os.makedirs(f"{PKG_DIR}/agent_bench", exist_ok=True)
pathlib.Path(f"{PKG_DIR}/agent_bench/core.py").write_text(core_py)
print("wrote core.py", len(core_py), "bytes")


### 3.2 · `registry.py` — L65 registries **+** L66 plugin discovery, merged

L68 caught a real bug: a "condensed" `registry.py` had silently dropped L66's
`discover_plugins()` / `load_plugins()`. Today we ship the **merged** version — class-based
registries (L67) *and* entry-point discovery (L66) in one file, so nothing is lost on the way
to the wheel.


In [ ]:
registry_py = r"""
from __future__ import annotations
import importlib.metadata as im

ENVIRONMENT_REGISTRY: dict = {}
AGENT_REGISTRY: dict = {}
SCORER_REGISTRY: dict = {}

def _register(reg, name, obj):
    if name in reg:
        raise ValueError(f"duplicate registration: {name!r} already in {reg}")
    reg[name] = obj
    return obj

def register_environment(name):
    def deco(cls):
        return _register(ENVIRONMENT_REGISTRY, name, cls)
    return deco

def register_agent(name):
    def deco(obj):
        return _register(AGENT_REGISTRY, name, obj)
    return deco

def register_scorer(name):
    def deco(fn):
        return _register(SCORER_REGISTRY, name, fn)
    return deco

def get_environment(name):
    # Registries store CLASSES -> fresh instance per call (L67: no cross-task state leak).
    return ENVIRONMENT_REGISTRY[name]()

def get_agent(name):
    return AGENT_REGISTRY[name]

def get_scorer(name):
    return SCORER_REGISTRY[name]

# ---- L66 plugin discovery: read-then-load, failures isolated per plugin ----
def discover_plugins(group="agent_bench.environments"):
    # Read-only probe of installed entry points. Does NOT import anything.
    try:
        eps = im.entry_points(group=group)
    except TypeError:  # older importlib API
        eps = im.entry_points().get(group, [])
    return [(ep.name, ep.value) for ep in eps]

def load_plugins(group="agent_bench.environments"):
    # Import each plugin; its @register_* decorator fires as a side effect.
    loaded, failed = [], []
    try:
        eps = im.entry_points(group=group)
    except TypeError:
        eps = im.entry_points().get(group, [])
    for ep in eps:
        try:
            ep.load()          # triggers registration
            loaded.append(ep.name)
        except Exception as e:  # one broken package must not crash the harness
            failed.append((ep.name, repr(e)))
    return {"loaded": loaded, "failed": failed}
"""
pathlib.Path(f"{PKG_DIR}/agent_bench/registry.py").write_text(registry_py)
print("wrote registry.py", len(registry_py), "bytes")


### 3.3 · `environments.py` — built-in envs, agent, scorers (L67)

`CalcEnv`, `FileEnv`, the deterministic `MockAgent` (with a `flake_rate` knob for testing
`pass@k`), and two scorers. All self-register on import.


In [ ]:
environments_py = r"""
from __future__ import annotations
import random
from .core import Trajectory, TaskResult
from .registry import register_environment, register_agent, register_scorer

@register_environment("calc")
class CalcEnv:
    name = "calc"
    def __init__(self):
        self._last = None
    def act(self, tool, args):
        if tool == "compute":
            self._last = eval(args["expr"], {"__builtins__": {}}, {})  # sandboxed literal math
            return self._last
        raise ValueError(f"unknown tool {tool}")
    def final_state(self):
        return self._last

@register_environment("file")
class FileEnv:
    name = "file"
    def __init__(self):
        self._fs = {}
    def act(self, tool, args):
        if tool == "write":
            self._fs[args["path"]] = args["content"]
            return "ok"
        if tool == "read":
            return self._fs.get(args["path"])
        raise ValueError(f"unknown tool {tool}")
    def final_state(self):
        return dict(self._fs)

class MockAgent:
    # Deterministic scripted agent. flake_rate injects controlled randomness for pass@k demos.
    name = "mock"
    def __init__(self, flake_rate=0.0, seed=0):
        self.flake_rate = flake_rate
        self._rng = random.Random(seed)
    def act(self, task, env):
        traj = Trajectory()
        if self._rng.random() < self.flake_rate:
            traj.final_state = "__FLAKE__"        # deliberate wrong answer
            return traj
        if task.env_name == "calc":
            expr = task.prompt.split("Compute:")[-1].split("=")[0].strip()
            obs = env.act("compute", {"expr": expr})
            traj.add("compute", {"expr": expr}, obs)
            traj.final_state = env.final_state()
        elif task.env_name == "file":
            env.act("write", {"path": "out.txt", "content": task.answer or ""})
            traj.add("write", {"path": "out.txt"}, "ok")
            traj.final_state = env.final_state()
        return traj

register_agent("mock")(MockAgent(flake_rate=0.0))

@register_scorer("exact_match")
def exact_match(task, traj):
    got = traj.final_state
    exp = task.answer
    try:
        return float(str(got).strip() == str(exp).strip())
    except Exception:
        return 0.0

@register_scorer("file_written")
def file_written(task, traj):
    fs = traj.final_state or {}
    return float(isinstance(fs, dict) and fs.get("out.txt") == (task.answer or ""))
"""
pathlib.Path(f"{PKG_DIR}/agent_bench/environments.py").write_text(environments_py)
print("wrote environments.py", len(environments_py), "bytes")


### 3.4 · `runner_async.py` — the async harness (L67)

`run_async` bounds concurrency with a `Semaphore` and isolates each attempt's failure into a
failed `TaskResult` (never lets an exception escape `gather`). `run_sync` is a thin
`asyncio.run` wrapper so scripts and the CLI have a blocking entry point.


In [ ]:
runner_py = r"""
from __future__ import annotations
import asyncio
from .core import TaskResult, pass_at_k
from .registry import get_environment, get_scorer

class AsyncBenchmarkRunner:
    def __init__(self, tasks):
        self.tasks = tasks

    async def _run_one(self, agent, task, sem):
        async with sem:
            try:
                env = get_environment(task.env_name)     # fresh instance per attempt (L67)
                loop = asyncio.get_event_loop()
                traj = await loop.run_in_executor(None, agent.act, task, env)
                score = get_scorer(task.scorer_name)(task, traj)
                return TaskResult(task.id, passed=score >= 1.0, score=score)
            except Exception as e:
                return TaskResult(task.id, passed=False, score=0.0, error=repr(e))

    async def run_async(self, agent, k=1, concurrency=8):
        sem = asyncio.Semaphore(concurrency)
        jobs = [self._run_one(agent, t, sem) for t in self.tasks for _ in range(k)]
        results = await asyncio.gather(*jobs)
        return results

    def run_sync(self, agent, k=1, concurrency=8):
        return asyncio.run(self.run_async(agent, k=k, concurrency=concurrency))

def summarize(results):
    n = len(results)
    passed = sum(r.passed for r in results)
    return {"n": n, "passed": passed, "pass_rate": (passed / n) if n else 0.0}
"""
pathlib.Path(f"{PKG_DIR}/agent_bench/runner_async.py").write_text(runner_py)
print("wrote runner_async.py", len(runner_py), "bytes")


### 3.5 · `cli.py` + `__init__.py` — the Typer frontend (L68)

Production CLI: a real console-script, `--version` eager callback reading
`importlib.metadata`, `--format table|json`. `Console(force_jupyter=False, no_color=True,
highlight=False)` — the L64 pitfall that bites any Rich CLI imported inside a notebook.


In [ ]:
init_py = '__version__ = "0.1.0"\n'
pathlib.Path(f"{PKG_DIR}/agent_bench/__init__.py").write_text(init_py)

cli_py = r"""
from __future__ import annotations
import json
import typer
from rich.console import Console
from rich.table import Table
from importlib.metadata import version, PackageNotFoundError
from .registry import ENVIRONMENT_REGISTRY, AGENT_REGISTRY, SCORER_REGISTRY, discover_plugins
from . import environments  # noqa: F401 - triggers built-in registration

console = Console(force_jupyter=False, no_color=True, highlight=False)  # L64 pitfall guard
app = typer.Typer(add_completion=True, help="agent-bench: a tiny agent benchmark harness.")

def _version_cb(value: bool):
    if value:
        try:
            console.print(f"agent-bench {version('agent-bench')}")
        except PackageNotFoundError:
            console.print('agent-bench (dev)')
        raise typer.Exit()

@app.callback()
def main(version: bool = typer.Option(False, '--version', callback=_version_cb, is_eager=True)):
    pass

@app.command('list-environments')
def list_environments(fmt: str = typer.Option('table', '--format')):
    names = sorted(ENVIRONMENT_REGISTRY)
    if fmt == 'json':
        console.print(json.dumps(names)); return
    t = Table('environment'); [t.add_row(n) for n in names]; console.print(t)

@app.command('list-scorers')
def list_scorers(fmt: str = typer.Option('table', '--format')):
    names = sorted(SCORER_REGISTRY)
    if fmt == 'json':
        console.print(json.dumps(names)); return
    t = Table('scorer'); [t.add_row(n) for n in names]; console.print(t)

@app.command('list-plugins')
def list_plugins():
    for name, val in discover_plugins():
        console.print(f"{name} -> {val}")

if __name__ == '__main__':
    app()
"""
pathlib.Path(f"{PKG_DIR}/agent_bench/cli.py").write_text(cli_py)
print("wrote __init__.py + cli.py")


## 4 · The integration test — **the load-bearing cell**

Install the package for real (`pip install -e .`), then run a benchmark end-to-end through
the *installed* modules — not the strings we just wrote to disk. This is the first time all
five modules execute together in one process. If any two drifted apart across L65–L69, it
surfaces **here**, before the wheel is built.

> L68's hard-won lesson: right after `pip install -e .` in a long-lived kernel, you must
> `site.addsitedir(...)` + `importlib.invalidate_caches()`, because the interpreter's
> `sys.path` was computed at startup and predates pip's `.pth` write.


In [ ]:
# 1) Write a minimal pyproject so `pip install -e .` works, then install.
pyproject = r"""
[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[project]
name = "agent-bench"
version = "0.1.0"
description = "A tiny, plugin-extensible agent benchmark harness."
readme = "README.md"
requires-python = ">=3.10"
license = {text = "MIT"}
authors = [{name = "Gourav Khanijoe"}]
dependencies = ["typer>=0.12", "pydantic>=2", "rich>=13"]

[project.optional-dependencies]
dev = ["pytest>=8", "build", "twine"]

[project.scripts]
agent-bench = "agent_bench.cli:app"

[project.entry-points."agent_bench.environments"]
# third-party plugins populate this table; agent-bench never registers itself here

[tool.hatch.build.targets.wheel]
packages = ["agent_bench"]
"""
pathlib.Path(f"{PKG_DIR}/pyproject.toml").write_text(pyproject)
pathlib.Path(f"{PKG_DIR}/LICENSE").write_text("MIT License\n\nCopyright (c) 2026 Gourav Khanijoe\n")
pathlib.Path(f"{PKG_DIR}/README.md").write_text("# agent-bench\n\nA tiny agent benchmark harness.\n")

r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", PKG_DIR],
                   capture_output=True, text=True)
print("pip install -e ->", "OK" if r.returncode == 0 else "FAILED")
if r.returncode != 0:
    print(r.stderr[-1500:])

# 2) Refresh sys.path so the just-installed package is importable in THIS kernel (L68 pitfall).
for p in {sysconfig.get_path("purelib"), site.getusersitepackages()}:
    site.addsitedir(p)
importlib.invalidate_caches()


In [ ]:
# 3) Import the INSTALLED modules (not the on-disk strings) and run end-to-end.
import agent_bench
from agent_bench.core import Task, pass_at_k
from agent_bench import environments            # registers built-ins
from agent_bench.runner_async import AsyncBenchmarkRunner, summarize
from agent_bench.registry import get_agent, ENVIRONMENT_REGISTRY, SCORER_REGISTRY

print("agent_bench version:", agent_bench.__version__)
print("environments:", sorted(ENVIRONMENT_REGISTRY))
print("scorers      :", sorted(SCORER_REGISTRY))

tasks = [
    Task(id="c1", category="calc", difficulty="easy",
         prompt="Compute: 2 + 3 =", env_name="calc", scorer_name="exact_match", answer="5"),
    Task(id="c2", category="calc", difficulty="easy",
         prompt="Compute: 6 * 7 =", env_name="calc", scorer_name="exact_match", answer="42"),
    Task(id="f1", category="file", difficulty="easy",
         prompt="Write hello to a file", env_name="file", scorer_name="file_written", answer="hello"),
]

runner = AsyncBenchmarkRunner(tasks)
agent = get_agent("mock")
results = runner.run_sync(agent, k=1, concurrency=4)
print("\\nend-to-end results:")
for res in results:
    print(f"  {res.task_id}: passed={res.passed} score={res.score} err={res.error}")
print("\\nsummary:", summarize(results))

INTEGRATION_OK = all(r.passed for r in results)
print("\\nINTEGRATION TEST:", "✅ PASS — all modules agree" if INTEGRATION_OK else "❌ FAIL")
assert INTEGRATION_OK, "modules drifted apart across L65-L69!"


### 4.1 · Prove the harness can measure flakiness — `pass@k` on a flaky agent

A launch-day benchmark that always reports 100% is a benchmark that isn't measuring anything.
Run a deliberately flaky agent and confirm `pass@5 > pass@1` — the harness genuinely
distinguishes reliable from unreliable agents.


In [ ]:
from agent_bench.environments import MockAgent

flaky = MockAgent(flake_rate=0.35, seed=7)
N = 10
per_task_attempts = {t.id: 0 for t in tasks}
per_task_correct = {t.id: 0 for t in tasks}

runner_n = AsyncBenchmarkRunner(tasks)
res_n = runner_n.run_sync(flaky, k=N, concurrency=8)
for r in res_n:
    per_task_attempts[r.task_id] += 1
    per_task_correct[r.task_id] += int(r.passed)

print(f"{'task':>5} {'n':>3} {'c':>3} {'pass@1':>7} {'pass@5':>7}")
for tid in per_task_attempts:
    n, c = per_task_attempts[tid], per_task_correct[tid]
    p1, p5 = pass_at_k(n, c, 1), pass_at_k(n, c, 5)
    print(f"{tid:>5} {n:>3} {c:>3} {p1:>7.2f} {p5:>7.2f}")
    assert p5 >= p1, "pass@k must be monotonic in k"
print("\\n✅ pass@5 >= pass@1 for every task — the harness measures reliability, not noise.")


## 5 · Build the wheel + `twine check` — the exact commands CI runs

`pip install -e .` proving the *source tree* works is **not** the same as the *packaging
config* being correct (L68 pitfall). Build a real wheel + sdist and run `twine check` on both
— the precise gate `ci.yml` enforces before any publish.


In [ ]:
# Build wheel + sdist from the package dir.
build = subprocess.run([sys.executable, "-m", "build", PKG_DIR],
                       capture_output=True, text=True)
print("python -m build ->", "OK" if build.returncode == 0 else "FAILED")
if build.returncode != 0:
    print(build.stdout[-1200:]); print(build.stderr[-1200:])

dist = pathlib.Path(f"{PKG_DIR}/dist")
artifacts = sorted(str(p) for p in dist.glob("*")) if dist.exists() else []
print("dist artifacts:", [pathlib.Path(a).name for a in artifacts])

# twine check both artifacts.
if artifacts:
    tw = subprocess.run([sys.executable, "-m", "twine", "check", *artifacts],
                        capture_output=True, text=True)
    print("\\ntwine check ->")
    print(tw.stdout.strip() or tw.stderr.strip())
    TWINE_OK = tw.returncode == 0 and "PASSED" in tw.stdout
else:
    TWINE_OK = False
print("\\nPACKAGING GATE:", "✅ PASS" if TWINE_OK else "❌ FAIL")


### 5.1 · Fresh-install verification — uninstall the editable, install the built wheel

The editable install can pass while the wheel is broken. Uninstall `-e`, install the actual
`.whl`, and re-run `--version` through the real console script. This proves the *packaging
config*, not just the source tree.


In [ ]:
wheel = next((a for a in artifacts if a.endswith(".whl")), None)
if wheel:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "agent-bench"],
                   capture_output=True, text=True)
    inst = subprocess.run([sys.executable, "-m", "pip", "install", "-q", wheel],
                          capture_output=True, text=True)
    print("install built wheel ->", "OK" if inst.returncode == 0 else "FAILED")
    for p in {sysconfig.get_path("purelib"), site.getusersitepackages()}:
        site.addsitedir(p)
    importlib.invalidate_caches()

    # Run --version through the actual installed console script if resolvable, else python -m.
    exe = shutil.which("agent-bench")
    cmd = [exe, "--version"] if exe else [sys.executable, "-m", "agent_bench.cli", "--version"]
    ver = subprocess.run(cmd, capture_output=True, text=True)
    print("console-script --version ->", (ver.stdout or ver.stderr).strip())
    FRESH_OK = "0.1.0" in (ver.stdout + ver.stderr)
else:
    FRESH_OK = False
print("FRESH-INSTALL VERIFICATION:", "✅ PASS" if FRESH_OK else "❌ FAIL")


## 6 · Portfolio launch — the one genuinely new idea

You already own a public repo (`paper-distiller`, L64). Launching a *second* one is not just
"repeat L64." Two coordination problems appear that a solo first-launch never had:

1. **Timing.** Two hard-launches in the same week split your attention and your audience's.
   The launch checklist below adds an explicit gate: **no `paper-distiller` hard-launch
   scheduled within ±1 week of this one.**
2. **Cross-promotion.** Each repo's README should point at the other. `agent-bench` can score
   *any* agent — including `paper-distiller`'s — so the link is substantive, not vanity.

Let's generate the launch-day files: a finalized cross-promoting README, a
`LAUNCH_CHECKLIST.md` with the timing gate, and a real `first_issue.md` sourced from the
L65–L69 homework backlog.


In [ ]:
# 6.1 — Finalized README with the cross-promotion callout (L69 draft -> launch version).
readme = textwrap.dedent('''\
# agent-bench

[![CI](https://github.com/gouravkhanijoe/agent-bench/actions/workflows/ci.yml/badge.svg)](https://github.com/gouravkhanijoe/agent-bench/actions/workflows/ci.yml)
[![PyPI](https://img.shields.io/pypi/v/agent-bench.svg)](https://pypi.org/project/agent-bench/)
[![Python](https://img.shields.io/pypi/pyversions/agent-bench.svg)](https://pypi.org/project/agent-bench/)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](LICENSE)

A tiny, plugin-extensible harness for benchmarking LLM agents. Task -> Environment -> Agent
-> Trajectory -> Scorer, with `pass@k`, async execution, and third-party environment plugins.

> **Part of a two-tool kit.** Pair with
> [**paper-distiller**](https://github.com/gouravkhanijoe/paper-distiller) — turn an arXiv id
> into a structured digest. `agent-bench` can score paper-distiller (or any agent) as a
> plugin environment.

## 30-second quickstart
```bash
pip install agent-bench
agent-bench list-environments
```
```python
from agent_bench.core import Task
from agent_bench.runner_async import AsyncBenchmarkRunner, summarize
from agent_bench.registry import get_agent

tasks = [Task(id="c1", category="calc", difficulty="easy",
              prompt="Compute: 2 + 3 =", env_name="calc",
              scorer_name="exact_match", answer="5")]
print(summarize(AsyncBenchmarkRunner(tasks).run_sync(get_agent("mock"))))
```

## Why not X?
| | manual pytest | one-off eval script | **agent-bench** |
|--|--|--|--|
| trajectory scoring | ✗ | partial | ✓ |
| `pass@k` | ✗ | ✗ | ✓ |
| async / concurrency | ✗ | ✗ | ✓ |
| third-party plugins | ✗ | ✗ | ✓ (`entry_points`) |

## Plugin gallery
| Plugin | What it adds |
|--------|--------------|
| `agent-bench-shell-plugin` | `ShellEnv` — sandboxed shell tasks (L66) |

## License
MIT
''')
pathlib.Path(f"{PKG_DIR}/README.md").write_text(readme)
print("README.md ->", len(readme), "bytes")


In [ ]:
# 6.2 — LAUNCH_CHECKLIST.md with the portfolio timing gate.
checklist = textwrap.dedent('''\
# agent-bench v0.1.0 — Launch Checklist

## Pre-flight (do NOT tag until every box is checked)
- [ ] `pytest` green locally
- [ ] `python -m build` + `twine check dist/*` both PASS
- [ ] Fresh-install verification: built wheel installs, console script `--version` works
- [ ] README quickstart copy-pasted into a clean venv and it runs
- [ ] LICENSE present (MIT)
- [ ] CI is GREEN on the commit you are about to tag  <-- tag-before-CI-green is the #1 killer
- [ ] **PORTFOLIO GATE: no paper-distiller hard-launch scheduled within +/- 1 week**

## Publish
- [ ] TestPyPI dry-run: `twine upload --repository testpypi dist/*`; `pip install` from it
- [ ] Register PyPI Trusted Publisher (OIDC) for this repo — no long-lived token
- [ ] `git tag v0.1.0 && git push origin v0.1.0`  (release.yml publishes on tag)
- [ ] Verify `pip install agent-bench` from real PyPI in a clean venv

## Post-launch (same day)
- [ ] File 3-5 good-first-issues from the L65-L69 homework backlog
- [ ] Add the cross-promotion snippet to paper-distiller's real README
- [ ] Submit to ONE awesome list (awesome-llm-eval / awesome-agents), NOT the same list as paper-distiller
- [ ] Announce on exactly ONE channel this week

## Explicitly NOT this week
- [ ] Second announcement channel (space them out)
- [ ] paper-distiller feature launch
''')
pathlib.Path(f"{PKG_DIR}/LAUNCH_CHECKLIST.md").write_text(checklist)
print("LAUNCH_CHECKLIST.md ->", len(checklist), "bytes")


In [ ]:
# 6.3 — A real good-first-issue sourced from the L65-L69 homework backlog.
first_issue = textwrap.dedent('''\
---
title: "Add a per-attempt timeout to AsyncBenchmarkRunner"
labels: ["good first issue", "enhancement"]
---

### Problem
`AsyncBenchmarkRunner._run_one` awaits `agent.act` with no timeout. A single hung agent
attempt can stall an entire benchmark run indefinitely.

### Proposed solution
Wrap the `run_in_executor` call in `asyncio.wait_for(..., timeout=task_timeout)` and, on
`TimeoutError`, return a failed `TaskResult(error="timeout")` instead of hanging. Add a
`--timeout` CLI option (default 60s) plumbed through `run_async`.

### Good first issue because
- Self-contained: one method + one CLI option
- Clear done-criteria: a test with a deliberately slow MockAgent returns a `timeout` result
- Touches the async pattern (L67) without needing the plugin or packaging machinery

### Pointers
- `agent_bench/runner_async.py` — `_run_one`
- Homework from Lesson 67 (per-attempt timeout via `wait_for`)
''')
pathlib.Path(f"{PKG_DIR}/first_issue.md").write_text(first_issue)
print("first_issue.md ->", len(first_issue), "bytes")
print("\\ngh command to file it:")
print('  gh issue create --title "Add a per-attempt timeout to AsyncBenchmarkRunner" \\\\')
print('    --label "good first issue,enhancement" --body-file first_issue.md')


## 7 · The launch runbook — the honest capability boundary

Everything above ran to completion inside this notebook. The steps below **cannot** — they
require *your* GitHub and PyPI identity and credentials. This is not a lesser capability an
assistant is withholding; it's that publishing under your name is, correctly, something only
you can authorize. Here is the exact, copy-pasteable sequence.

```bash
# 0) From the agent-bench package directory, with a clean git tree:
cd agent_bench_pkg
git init && git add -A && git commit -m "agent-bench v0.1.0"

# 1) Create the GitHub repo and push (uses the gh CLI, already authed as you):
gh repo create gouravkhanijoe/agent-bench --public --source=. --push

# 2) WATCH CI GO GREEN before doing anything else:
gh run watch          # do not proceed until this is green — tag-before-green kills launches

# 3) TestPyPI dry-run (catch metadata problems before the real index):
python -m build
twine upload --repository testpypi dist/*
pip install -i https://test.pypi.org/simple/ agent-bench   # in a clean venv

# 4) Register the PyPI Trusted Publisher (OIDC) — browser, one time:
#    pypi.org -> your project -> Publishing -> add GitHub Actions publisher
#    (owner=gouravkhanijoe, repo=agent-bench, workflow=release.yml). No token stored.

# 5) Tag -> release.yml publishes to real PyPI automatically:
git tag v0.1.0 && git push origin v0.1.0

# 6) Verify the real thing from a clean venv:
pip install agent-bench && agent-bench --version

# 7) File the first good-first-issue (see §6.3):
gh issue create --title "Add a per-attempt timeout to AsyncBenchmarkRunner" \
  --label "good first issue,enhancement" --body-file first_issue.md
```

**Portfolio step (do this too, but not the same day as a paper-distiller launch):** open a PR
adding the cross-promotion snippet to `paper-distiller`'s README, and submit `agent-bench` to
a *different* awesome list than the one paper-distiller is on.


In [ ]:
# Write the runbook and CI/release workflows to disk so they ship with the repo.
os.makedirs(f"{PKG_DIR}/.github/workflows", exist_ok=True)

ci_yml = textwrap.dedent('''\
name: CI
on: [push, pull_request]
jobs:
  test:
    runs-on: ubuntu-latest
    strategy:
      matrix:
        python-version: ["3.10", "3.11", "3.12"]
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: {python-version: "${{ matrix.python-version }}"}
      - run: pip install -e ".[dev]"
      - run: pytest -q
      - run: python -m build
      - run: twine check dist/*
''')
pathlib.Path(f"{PKG_DIR}/.github/workflows/ci.yml").write_text(ci_yml)

release_yml = textwrap.dedent('''\
name: Release
on:
  push:
    tags: ["v*"]
permissions:
  id-token: write        # OIDC Trusted Publishing — no PyPI token stored
jobs:
  publish:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: {python-version: "3.12"}
      - run: pip install build && python -m build
      - uses: pypa/gh-action-pypi-publish@release/v1
''')
pathlib.Path(f"{PKG_DIR}/.github/workflows/release.yml").write_text(release_yml)
print("wrote .github/workflows/{ci,release}.yml")


## 8 · Launch-day pitfalls (10)

| # | Pitfall | Fix |
|---|---------|-----|
| 1 | **Tag before CI is green** | `gh run watch` until green; *then* `git tag` |
| 2 | Skipping the cross-module integration test | Run §4 in one kernel before building |
| 3 | Editable install passes, wheel is broken | §5.1 fresh-install from the built `.whl` |
| 4 | `twine check` PASS ≠ "the package works" | Also import + run from a clean venv |
| 5 | **Two hard-launches the same week** | Portfolio gate: ±1 week from paper-distiller |
| 6 | Stale README / dead cross-promo link | Verify both repos' links resolve at launch |
| 7 | Zero issues filed at launch | File 3–5 good-first-issues from the backlog |
| 8 | Same awesome list for both repos | Target `awesome-llm-eval` vs paper-distiller's list |
| 9 | Storing a long-lived PyPI token | OIDC Trusted Publishing, `id-token: write` |
| 10 | "Production ready" before a real tagged CI run | Ship v0.1.0; let CI prove it on the tag |


## 9 · Verification checklist

Every launch gate, asserted — not narrated.


In [ ]:
checks = []
def check(name, cond):
    checks.append((name, bool(cond)))
    print(("✅" if cond else "❌"), name)

# Files present
for rel in ["pyproject.toml", "README.md", "LICENSE", "LAUNCH_CHECKLIST.md",
            "first_issue.md", ".github/workflows/ci.yml", ".github/workflows/release.yml",
            "agent_bench/core.py", "agent_bench/registry.py", "agent_bench/environments.py",
            "agent_bench/runner_async.py", "agent_bench/cli.py", "agent_bench/__init__.py"]:
    check(f"file exists: {rel}", pathlib.Path(f"{PKG_DIR}/{rel}").exists())

# Load-bearing gates from earlier cells
check("integration test passed (all modules agree)", INTEGRATION_OK)
check("packaging gate: twine check PASSED", TWINE_OK)
check("fresh-install of built wheel works", FRESH_OK)

# Content assertions
check("README cross-promotes paper-distiller",
      "paper-distiller" in pathlib.Path(f"{PKG_DIR}/README.md").read_text())
check("checklist has the +/- 1 week portfolio gate",
      "+/- 1 week" in pathlib.Path(f"{PKG_DIR}/LAUNCH_CHECKLIST.md").read_text())
check("release.yml uses OIDC (id-token: write), no token",
      "id-token: write" in pathlib.Path(f"{PKG_DIR}/.github/workflows/release.yml").read_text())

passed = sum(c for _, c in checks)
print(f"\\n{passed}/{len(checks)} checks passed")
assert passed == len(checks), "launch verification failed"
print("🚀 ALL LAUNCH GATES GREEN — agent-bench v0.1.0 is ready to ship.")


## 10 · Summary, homework, and where the curriculum goes next

### What you shipped today

| Concept | Takeaway |
|---------|----------|
| Cross-module integration test | Modules validated in isolation *will* drift; prove they agree in one kernel |
| Build ≠ editable install | `twine check` + fresh-install from the `.whl` catch packaging bugs `-e` hides |
| OIDC Trusted Publishing | Tag-triggered release, no long-lived token |
| Portfolio launch timing | Don't hard-launch two repos within ±1 week |
| Substantive cross-promotion | Each README links the other; `agent-bench` can score `paper-distiller` |
| Honest capability boundary | Publishing under *your* identity is yours to authorize, by design |
| Good-first-issue sourcing | Turn homework backlog into a contributor on-ramp at launch |

### 🎓 Phase 7 — COMPLETE (L65–L70, 6 lessons)

| # | Lesson | Core idea |
|---|--------|-----------|
| 65 | Kickoff | notebook demo → library; plugin-registry pattern |
| 66 | Real plugins | `entry_points()` third-party environment discovery |
| 67 | Async execution | `asyncio.gather` + `Semaphore`, fresh-instance-per-task |
| 68 | CLI + PyPI | console script, `--version`, wheel/sdist, OIDC |
| 69 | OSS growth (round 2) | README/CONTRIBUTING funnel, portfolio cross-promo |
| 70 | **Launch Day** | integration test → wheel → runbook → **ship** |

**Curriculum now stands at 70 lessons across 7 phases.** You have shipped *two* flagship OSS
tools (`paper-distiller`, `agent-bench`) that reference each other — a genuine portfolio.

### 📝 Homework
1. Run the §7 runbook for real: create the repo, watch CI green, publish v0.1.0 via OIDC.
2. Implement the §6.3 good-first-issue yourself (per-attempt timeout) as the first merged PR.
3. Open the cross-promotion PR on `paper-distiller`'s real README.
4. Submit `agent-bench` to `awesome-llm-eval` (a *different* list than paper-distiller's).
5. Wire `agent-bench` as a 4th CI job that scores `paper-distiller` itself — closing the loop
   between your two repos.

### 🔭 Phase 8 preview (tentative — adapts to your questions)
With two shipped tools behind you, Phase 8 shifts from *building your own* to *operating agents
in production and reading the field*. Three candidate tracks:
- **8A — Observability & evals in prod:** tracing (OpenTelemetry for LLMs), online eval,
  cost/latency dashboards, regression alerting on live traffic.
- **8B — Multi-agent orchestration:** planner/worker patterns, message passing, shared memory,
  and where they actually beat a single well-scaffolded agent (and where they don't).
- **8C — Research literacy:** reproduce one recent agent paper end-to-end using your own
  `agent-bench` harness as the measurement layer.

Absent other input, the next run defaults to **8A** (it compounds directly on the two tools
you now operate). Ask a question on any run to redirect this.

> 🎉 **Congratulations, Gourav.** Seventy lessons ago you were a Java engineer with no ML
> background. You now have two public, installable, tested, documented AI tooling projects and
> the vocabulary to reason about agents from data model to launch. That *is* the proof of
> worth you set out to build.
